### Minizinc

[Vi

#### Clustering Problem
Gegeben eine Reihe von Punkten, wir wollen sie so in k Cluster aufteilen, dass die Punkte in einem Cluster untereinander nicht weiter als 5 entfernt sind und wir wollen die minimalen Abstände zwischen zwei Punkten in verschiedenen Clustern maximieren (die Cluster sollen möglichst weit voneinander entfernt sein).

<img src='cluster1.png' width='200'>
<img src='cluster2.png' width='500'>

Für jeden Punkt brauchen wir eine Zuordnung zu einem Cluster. Eine Lösung können wir modellieren als ein array, dass jedem Punkt die Nummer eines Clusters zuordnet.

In [ ]:
array[1..nSPOT] of var 1..k: x;

In [ ]:
x = [1,2,1,3]
x = [3,1,3,2]

Beide Lösungen beschreiben dieselbe Partition. Wir sollten die Modellierung so machen, das eine Partition von genau einer Lösung in unserem Modell beschrieben wird.


Hier liegt eine *value symmetry* vor: die Werte 1..k sind symmetrisch zueinander, d.h. uns ist es egal welche Nummer welcher Kreis bekommt.

Wenn eine Partition aus den drei Mengen {2}, {4}, und {1,3} besteht, dann sortieren wir die Mengen nach ihrem kleinsten Element:
{1,3}, {2}, {4} und ordnen dann die Nummer des Clusters zu. Dadurch wird nur noch die Zuordnung x = [1,2,1,3]  möglich.
Unser constraint verlangt, dass die kleinste Zahl in partition i kleiner ist als die kleinste Zahl in partition i+1.

In [ ]:
set of int: SPOT = 1..nSpot;
constraint forall (i in 1..k-1)
    (min([j | j in SPOT where x[j] = i]) < min([j | j in SPOT where x[j] = i+1]));

#### dzn

In [5]:
%%writefile test.dzn
nSpot = 8;
k = 3; %  

maxSep = 5.0;

dist = 
[|0.000, 1.414, 2.236, 3.606, 5.831, 6.403, 6.325, 7.616 |
1.414, 0.000, 1.000, 4.123, 6.325, 6.708, 5.831, 7.211 |
2.236, 1.000, 0.000, 4.000, 6.083, 6.325, 5.000, 6.403 |
3.606, 4.123, 4.000, 0.000, 2.236, 2.828, 4.123, 5.000 |
5.831, 6.325, 6.083, 2.236, 0.000, 1.000, 4.243, 4.472 |
6.403, 6.708, 6.325, 2.828, 1.000, 0.000, 3.606, 3.606 |
6.325, 5.831, 5.000, 4.123, 4.243, 3.606, 0.000, 1.414 |
7.616, 7.211, 6.403, 5.000, 4.472, 3.606, 1.414, 0.000 |];

Overwriting test.dzn


#### mzn

In [7]:
%%writefile test.mzn
int: nSpot;
set of int: SPOT = 1..nSpot;

int: k;
set of int: CLUSTER = 1..k;

float: maxSep;
array[SPOT, SPOT] of float: dist;

array[SPOT] of var CLUSTER: x;  % clustering decisions

constraint forall(i,j in SPOT where x[i] = x[j])(dist[i, j] <= maxSep); % intra-cluster

constraint forall (i in 1..k-1)
    (min([j | j in SPOT where x[j] = i]) < min([j | j in SPOT where x[j] = i+1]));

float: maxdist = max([dist[i, j] | i, j in SPOT]); % auxiliary variable
var float: obj = min(i, j in SPOT where i < j)(if x[i] = x[j] then maxdist else dist[i, j] endif); % inter-cluster
solve maximize obj;

output[show(x), ", obj = ", show(obj)];

Overwriting test.mzn


Wir können den constraint für die value symmetrie ersetzen durch einen global constraint, der genau für diesen Zweck geschaffen ist.
Die Werte in der Liste CLUSTER dürfen in x nur in der Reihenfolge „zum ersten Mal auftauchen“ auftauchen, wie in CLUSTER vorgegeben.

In [ ]:
include "globals.mzn";
constraint value_precede_chain(CLUSTER, x);

Die optimale Lösung besteht aus 2 Punkten. Wenn wir genau drei Punkte haben wollen, dann fügen wir einen constraint hinzu, der dafür sorgt, dass jedes der k Cluster mindestens einen Bestandtteil hat

In [ ]:
constraint global_cardinality_low_up_closed(x,
  [i | i in CLUSTER], [1 | i in CLUSTER],
  [nSpot-k+1 | i in CLUSTER]);

Zur Erinnerung: die allgemeine Form des constraints ist

In [ ]:
global_cardinality_low_up_closed(x, values, low, up)

Für jeden Wert v in values gilt:
Die Anzahl der Variablen in x, die den Wert v annehmen, liegt zwischen low[v] und up[v]. Und andere Werte dürfen nicht vorkommen.

Wenn k nicht vorgeben ist, können wir k auf die Anzahl der Punkte setzen, in der Hoffnung, dass die Laufzeit erträglich bleibt.